# SQL Interview Practice — Day 17
## 10 Real Interview-Style Questions Found Online

**Level:** Data Analyst — 2–3 years experience  
Questions are paraphrased from online interview/problem sources. No solutions included.

In [90]:
import pandas as pd

customers = pd.DataFrame({
    "customer_id":[1,2,3,4,5,6,7,8],
    "customer_name":["Aarav","Diya","Kabir","Ishita","Rohan","Meera","Vivaan","Anaya"],
    "city":["Delhi","Mumbai","Pune","Delhi","Bengaluru","Mumbai","Pune","Delhi"]
})

orders = pd.DataFrame({
    "order_id":[1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013],
    "customer_id":[1,2,1,3,4,2,5,1,6,5,2,3,7],
    "order_date":pd.to_datetime(["2025-01-05","2025-01-10","2025-02-12","2025-02-20","2025-03-01","2025-03-18","2025-04-02","2025-04-15","2025-05-07","2025-05-20","2025-06-10","2025-06-15","2025-07-01"]),
    "amount":[1200,2500,1800,3200,900,4100,1500,2700,3600,2200,2800,3500,1700],
    "status":["Completed","Completed","Cancelled","Completed","Completed","Completed","Completed","Completed","Completed","Cancelled","Completed","Completed","Completed"]
})

purchases = pd.DataFrame({
    "user_id":[1,1,1,2,2,3,3,3,4,4,5,5],
    "purchase_id":[101,102,103,201,202,301,302,303,401,402,501,502],
    "purchase_date":pd.to_datetime(["2025-01-03","2025-01-15","2025-02-10","2025-01-05","2025-02-18","2025-01-01","2025-01-20","2025-03-05","2025-02-01","2025-03-10","2025-01-12","2025-04-01"]),
    "amount":[500,700,900,400,800,300,600,1200,450,950,250,1100]
})

logins = pd.DataFrame({
    "user_id":[1,1,1,1,2,2,3,3,3,4,4,4],
    "login_date":pd.to_datetime(["2025-06-01","2025-06-02","2025-06-03","2025-06-05","2025-06-01","2025-06-03","2025-06-01","2025-06-02","2025-06-03","2025-06-01","2025-06-03","2025-06-04"])
})

employees = pd.DataFrame({
    "employee_id":[101,102,103,104,105,106,107,108],
    "employee_name":["Amit","Neha","Rahul","Priya","Vikas","Sneha","Arjun","Pooja"],
    "manager_id":[None,101,101,103,103,105,101,103],
    "department_id":[10,10,20,20,30,30,10,20],
    "salary":[95000,78000,72000,88000,65000,82000,110000,76000]
})

search_results = pd.DataFrame({
    "query_id":[1,1,1,2,2,2,3,3],
    "result_id":[101,102,103,201,202,203,301,302],
    "rating":[5,2,1,4,2,1,2,2]
})

projects = pd.DataFrame({
    "project_id":[1,2,3,4],
    "project_name":["Alpha","Beta","Gamma","Delta"],
    "budget":[100000,150000,90000,200000]
})

project_spend = pd.DataFrame({
    "project_id":[1,1,2,2,3,4,4],
    "spend":[30000,45000,80000,50000,95000,60000,70000]
})

## Q1 — For each user, find their third purchase chronologically. Return user_id, purchase_id, purchase_date and amount. Use a window function.

In [91]:
import pandasql
from pandasql import sqldf
sqldf("""
    with my_cte as (
    select user_id , purchase_id , purchase_date , amount , rn
        from (select user_id , purchase_id , purchase_date , amount ,
                row_number()
                over(partition by user_id order by purchase_date) as rn
                from purchases)   
    )


select * from my_cte
where rn = 3







""")




,user_id,purchase_id,purchase_date,amount,rn
0,1,103,2025-02-10 00:00:00.000000,900,3
1,3,303,2025-03-05 00:00:00.000000,1200,3


## Q2 — Find users who made at least two purchases. Return user_id and purchase_count.

In [92]:
sqldf("""
select user_id , count(purchase_id) as count
from purchases
group by user_id
having count(*) >= 2



""")

,user_id,count
0,1,3
1,2,2
2,3,3
3,4,2
4,5,2


## Q3 — Find employees earning more than their manager. Return employee name/salary and manager name/salary. Use a SELF JOIN.

In [93]:
sqldf("""
select e.employee_name , e.salary , m.employee_name as manager_name , m.salary as manager_salary
from employees e
join employees m on m.employee_id = e.manager_id
where e.employee_name > m.employee_name

""")

,employee_name,salary,manager_name,manager_salary
0,Neha,78000,Amit,95000
1,Rahul,72000,Amit,95000
2,Vikas,65000,Rahul,72000
3,Arjun,110000,Amit,95000


In [94]:
employees

,employee_id,employee_name,manager_id,department_id,salary
0,101,Amit,NaN,10,95000
1,102,Neha,101.0,10,78000
2,103,Rahul,101.0,20,72000
3,104,Priya,103.0,20,88000
4,105,Vikas,103.0,30,65000
5,106,Sneha,105.0,30,82000
6,107,Arjun,101.0,10,110000
7,108,Pooja,103.0,20,76000


## Q4 — Find the top 3 salaries in every department. Tied salaries must receive the same rank.

In [95]:
sqldf("""
select department_id , salary ,rn
    from (select department_id , salary,
            dense_rank()
            over(partition by department_id order by salary desc) as rn
            from employees) t
where rn <= 3


""")

,department_id,salary,rn
0,10,110000,1
1,10,95000,2
2,10,78000,3
3,20,88000,1
4,20,76000,2
5,20,72000,3
6,30,82000,1
7,30,65000,2


## Q5 — Find customers who have never placed a completed order. Return customer_id, customer_name and city.

In [96]:
sqldf("""
SELECT
    c.customer_id,
    c.customer_name,
    c.city
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
    AND o.status = 'Completed'
WHERE o.order_id IS NULL;

""")

,customer_id,customer_name,city
0,8,Anaya,Delhi


## Q6 — For every user, find the longest streak of consecutive login days. Use a gaps-and-islands/window-function approach.

In [100]:
sqldf("""
WITH cte AS (
    SELECT
        user_id,
        login_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY login_date
        ) AS rn
    FROM logins
),

islands AS (
    SELECT
        user_id,
        login_date,
        DATE(login_date, '-' || rn || ' days') AS grp
    FROM cte
),

streaks AS (
    SELECT
        user_id,
        COUNT(*) AS streak_length
    FROM islands
    GROUP BY user_id, grp
)

SELECT
    user_id,
    MAX(streak_length) AS longest_streak
FROM streaks
GROUP BY user_id
ORDER BY user_id;



""")

,user_id,longest_streak
0,1,3
1,2,1
2,3,3
3,4,2


## Q7 — Calculate monthly completed revenue and month-over-month percentage growth. Return month, revenue, previous revenue and growth percentage.

In [102]:
sqldf("""
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', order_date) AS month,
        SUM(amount) AS revenue
    FROM orders
    WHERE status = 'Completed'
    GROUP BY strftime('%Y-%m', order_date)
),

growth_cte AS (
    SELECT
        month,
        revenue,
        LAG(revenue) OVER (ORDER BY month) AS previous_revenue
    FROM monthly_revenue
)

SELECT
    month,
    revenue,
    previous_revenue,
    ROUND(
        100.0 * (revenue - previous_revenue)
        / previous_revenue,
        2
    ) AS growth_percentage
FROM growth_cte;


""")

,month,revenue,previous_revenue,growth_percentage
0,2025-01,3700,NaN,NaN
1,2025-02,3200,3700.0,-13.51
2,2025-03,5000,3200.0,56.25
3,2025-04,4200,5000.0,-16.00
4,2025-05,3600,4200.0,-14.29
5,2025-06,6300,3600.0,75.00
6,2025-07,1700,6300.0,-73.02


## Q8 — For each search query, determine whether all returned results have a rating below 3, then calculate the overall percentage of such queries.

In [99]:
sqldf("""
SELECT
    ROUND(
        100.0 * COUNT(
            CASE WHEN max_rating < 3 THEN 1 END
        ) / COUNT(*),
        2
    ) AS percentage
FROM (
    SELECT
        query_id,
        MAX(rating) AS max_rating
    FROM search_results
    GROUP BY query_id
) t;


""")

,percentage
0,33.33


In [98]:
search_results

,query_id,result_id,rating
0,1,101,5
1,1,102,2
2,1,103,1
3,2,201,4
4,2,202,2
5,2,203,1
6,3,301,2
7,3,302,2


## Q9 — Calculate total spending for each project and classify it as Over Budget when spending exceeds budget, otherwise Within Budget.

In [68]:
sqldf("""
select p.project_id ,p.project_name ,p.budget, sum(s.spend),
case 
    when spend > budget then 'exceeds budget'
    else 'Within Budget'
end as buket 
from projects p
left join project_spend s
on p.project_id = s.project_id
group by p.project_id


""")

,project_id,project_name,budget,sum(s.spend),buket
0,1,Alpha,100000,75000,Within Budget
1,2,Beta,150000,130000,Within Budget
2,3,Gamma,90000,95000,exceeds budget
3,4,Delta,200000,130000,Within Budget


## Q10 — Find the latest purchase for every user. If multiple purchases occur on the same date, use purchase_id as the tie-breaker.

In [88]:
sqldf("""
WITH my_cte AS (
    SELECT
        user_id,
        purchase_id,
        purchase_date,
        amount,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY purchase_date DESC, purchase_id DESC
        ) AS rn
    FROM purchases
)

SELECT
    user_id,
    purchase_id,
    purchase_date,
    amount
FROM my_cte
WHERE rn = 1;
""")

,user_id,purchase_id,purchase_date,amount
0,1,103,2025-02-10 00:00:00.000000,900
1,2,202,2025-02-18 00:00:00.000000,800
2,3,303,2025-03-05 00:00:00.000000,1200
3,4,402,2025-03-10 00:00:00.000000,950
4,5,502,2025-04-01 00:00:00.000000,1100


### Sources

Questions are paraphrased from recurring interview/problem patterns found online, including Interview Query's Data Analyst SQL interview guide, LeetCode SQL interview discussions, and DataCamp's 2026 SQL interview preparation material.